In [0]:
from pyspark.sql.functions import split, col, regexp_replace, to_date, date_format

spark.sql("CREATE SCHEMA IF NOT EXISTS shoplive.silver")

customers_df = spark.table("shoplive.bronze.customers")
orders_df = spark.table("shoplive.bronze.orders")
products_df = spark.table("shoplive.bronze.products")
events_df = spark.table("shoplive.bronze.events")

print(f"Loaded - Customers: {customers_df.count():,} | Orders: {orders_df.count():,} | Products: {products_df.count():,} | Events: {events_df.count():,}")

In [0]:
clean_customers_df = customers_df.withColumn("first_name", split(col("name"), " ")[0]).withColumn("second_name", split(col("name"), " ")[1]).drop("name").withColumn("email", regexp_replace(col("email"), "(?<=.{2})[^@]+(?=@)", "***")).dropDuplicates(["customer_id"])

clean_orders_df = orders_df.dropDuplicates(["order_id"]).withColumn("order_date", to_date(col("order_ts"))).withColumn("order_time", date_format(col("order_ts"), "HH:mm:ss")).drop("order_ts")

clean_products_df = products_df.dropDuplicates(["product_id"])

clean_events_df = events_df.dropDuplicates(["event_id"]).withColumn("event_date", to_date(col("event_ts"))).withColumn("event_time", date_format(col("event_ts"), "HH:mm:ss")).drop("event_ts")

print(f"Cleaned - Customers: {clean_customers_df.count():,} | Orders: {clean_orders_df.count():,} | Products: {clean_products_df.count():,} | Events: {clean_events_df.count():,}")

In [0]:
clean_customers_df.write.mode("overwrite").saveAsTable("shoplive.silver.customers")
clean_orders_df.write.mode("overwrite").saveAsTable("shoplive.silver.orders")
clean_products_df.write.mode("overwrite").saveAsTable("shoplive.silver.products")
clean_events_df.write.mode("overwrite").saveAsTable("shoplive.silver.events")

print("All silver tables created successfully")